In [8]:
import warnings
warnings.filterwarnings('ignore')

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro, ttest_ind, mannwhitneyu, chi2_contingency
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# ── Reproducibility
SEED = 42
np.random.seed(SEED)

# ── Plotting style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
COLORS = sns.color_palette('husl', 2)

print('All imports OK')

All imports OK


In [9]:
FILE = 'X-cell MF 1 without test .xlsx'

# Load with MultiIndex columns (row 0 = group label, row 1 = actual column names)
raw = pd.read_excel(FILE, header=[0, 1], index_col=None)

# Flatten MultiIndex columns — keep only the second level and deduplicate
seen = {}
flat_cols = []
for col in raw.columns:
    name = str(col[1]).strip()
    seen[name] = seen.get(name, 0) + 1
    if seen[name] == 1:
        flat_cols.append(name)
    else:
        flat_cols.append(f"{name}_{seen[name]}")

raw.columns = flat_cols

# Drop the first row if it's a duplicate header (merged cells)
if raw.iloc[0].astype(str).str.lower().isin(['patient data', 'age', 'sex']).sum() > 3:
    raw = raw.iloc[1:].reset_index(drop=True)

print(f'Data shape: {raw.shape}')
print(f'Number of columns: {len(raw.columns)}')

Data shape: (499, 35)
Number of columns: 35


In [ ]:
# ── Canonical rename map
RENAME = {
    'Patient Data': 'patient_id',
    'Age': 'age',
    'Sex': 'sex',
    'no. of bx': 'num_bx',
    'Diagnosis': 'diagnosis',
    'Duration': 'duration_raw',
    'Course': 'course',
    'New, FU, Rec': 'visit_type',
    'Prev. TTT': 'prev_ttt',
    'Type of TTT': 'type_ttt',
    'Current TTT': 'current_ttt',
    'Type of current TTT': 'type_current_ttt',
    'Symptomatic': 'symptomatic',
    'Symptoms': 'symptoms',
    'Head and Neck': 'site_head_neck',
    'UL': 'site_ul',
    'LL': 'site_ll',
    'Trunk': 'site_trunk',
    'Buttocks': 'site_buttocks',
    'of BX 1 site': 'bx1_site',
    'of BX 2 site': 'bx2_site',
    'of BX 1 morph': 'bx1_morph',
    'of BX 2 morph': 'bx2_morph',
    'Stage of MF': 'stage_mf',
    'color': 'color',
    'Macules ': 'macules',
    ' Patch ': 'patch',
    'Papules ': 'papules',
    'Plaque': 'plaque',
    'nodule': 'nodule',
    'others': 'others',
    'others_2': 'others_2',
    'scales': 'scales',
    'Type of MF': 'type_mf',
}

# Strip and rename
raw.columns = [c.strip() for c in raw.columns]
raw = raw.rename(columns={k.strip(): v for k, v in RENAME.items()})

# Drop unnecessary columns
DROP_COLS = ['prev_ttt', 'type_ttt', 'type_current_ttt', 'type_mf',
             'others', 'others_2', 'patient_id', 'Type of MF.1', 'symptoms', 'num_bx', 'current_ttt']
raw.drop(columns=[c for c in DROP_COLS if c in raw.columns], inplace=True)

df = raw.copy()
print(f'After preprocessing: {df.shape}')

After preprocessing: (499, 25)


In [11]:
# ── Parse duration to numeric (months)
def parse_duration(val):
    """Convert '10 Y', '6 M', '2 W', n/a, NaN → float months."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower().replace('\\', '').replace('/', '')
    if s in ('na', 'n/a', 'na', ''):
        return np.nan
    # Extract leading number and unit letter
    m = re.match(r'([\d\.]+)\s*([ymwdYMWD])', s)
    if not m:
        # Fallback: try to parse any number
        nums = re.findall(r'[\d\.]+', s)
        return float(nums[0]) if nums else np.nan
    num, unit = float(m.group(1)), m.group(2).lower()
    return {'y': num * 12, 'm': num, 'w': num / 4.33, 'd': num / 30}.get(unit, np.nan)

df['duration_months'] = df['duration_raw'].apply(parse_duration)
df.drop(columns=['duration_raw'], inplace=True)

print('Duration parsing complete')

Duration parsing complete


In [ ]:
# ── Helper function to normalize free-text fields
def norm(s, mapping=None):
    """Lowercase, strip, optional remap."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lower()
    if mapping:
        for pat, rep in mapping.items():
            if re.fullmatch(pat, s):
                return rep
    return s

# Create binary target variable: 1 for MF, 0 for Non-MF
df['diagnosis_target'] = (df['diagnosis'].apply(lambda x: norm(x)) == 'mf').astype(int)

# Normalize categorical columns
# df['current_ttt'] = df['current_ttt'].fillna('no')
# df['current_ttt'] = df['current_ttt'].apply(
#     lambda x: 'no' if str(x).strip().lower() in ('n/a', 'na', 'nan', '') else str(x).strip().lower()
# )

# Handle absence of second biopsy as its own category
df['bx2_site'] = df['bx2_site'].fillna('no_bx_site')
df['bx2_morph'] = df['bx2_morph'].fillna('no_bx_morph')
df['bx1_site'] = df['bx1_site'].fillna('no_bx_site')
df['bx1_morph'] = df['bx1_morph'].fillna('no_bx_morph')

# Map visit_type
VISIT_MAP = {r'new': 'New', r'fu|follow.?up': 'FU', r'rec|recurrent': 'Rec'}
def map_visit(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    for pat, val in VISIT_MAP.items():
        if re.fullmatch(pat, s):
            return val
    return np.nan

df['visit_type'] = df['visit_type'].apply(map_visit)

# Normalize course
df['course'] = df['course'].apply(lambda x: norm(x, {r'progressive': 'progressive',
                                                        r'stationary': 'stationary'}))

# Normalize biopsy sites
for col in ['bx1_site', 'bx2_site']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: norm(x, {r'buttocks?': 'buttocks', r'n/?a': np.nan}))

# Normalize yes/no columns to 0/1 (NaN for missing)
YES_NO_COLS = ['site_head_neck', 'site_ul', 'site_ll', 'site_trunk',
               'site_buttocks', 'symptomatic', 'macules', 'patch',
               'papules', 'plaque', 'nodule', 'scales']
for col in YES_NO_COLS:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: 1 if str(x).strip().lower() in ('yes', 'y') else
                      (0 if str(x).strip().lower() in ('no', 'n') else np.nan)
        )

# Normalize sex
df['sex'] = df['sex'].apply(lambda x: norm(x, {r'male|m': 'male', r'female|f': 'female'}))

# Parse num_bx
if 'num_bx' in df.columns:
    df['num_bx'] = df['num_bx'].apply(
        lambda x: float(re.findall(r'[\d]+', str(x))[0])
        if re.findall(r'[\d]+', str(x)) else np.nan
    )

# Parse age
df['age'] = pd.to_numeric(df['age'], errors='coerce')

print('Preprocessing complete')
print(f'Target variable distribution:')
print(df['diagnosis_target'].value_counts())

Preprocessing complete
Target variable distribution:
diagnosis_target
1    353
0    146
Name: count, dtype: int64


In [13]:
# ── Encode categorical features for analysis
CAT_COLS = ['sex', 'course', 'color', 'bx1_site', 'bx2_site', 'bx1_morph', 'bx2_morph', 'current_ttt', 'visit_type']
CAT_COLS = [c for c in CAT_COLS if c in df.columns]

encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    non_null = df[col].notna()
    df.loc[non_null, col] = le.fit_transform(df.loc[non_null, col].astype(str))
    df[col] = pd.to_numeric(df[col], errors='coerce')
    encoders[col] = le

print('Categorical encoding complete')
print(f'\nDataset shape: {df.shape}')
print(f'Data types:\n{df.dtypes}')

Categorical encoding complete

Dataset shape: (499, 26)
Data types:
age                 float64
sex                   int64
diagnosis            object
course              float64
visit_type            int64
current_ttt           int64
symptomatic           int64
site_head_neck      float64
site_ul             float64
site_ll             float64
site_trunk          float64
site_buttocks         int64
bx1_site              int64
bx2_site              int64
bx1_morph             int64
bx2_morph             int64
stage_mf             object
color               float64
macules               int64
patch                 int64
papules               int64
plaque              float64
nodule                int64
scales                int64
duration_months     float64
diagnosis_target      int32
dtype: object


In [14]:
# ── Verification of Encoding Mappings
print("--- ENCODING MAPPINGS ---\n")

all_mappings = {}

# 1. CAT_COLS mappings from LabelEncoder
for col, le in encoders.items():
    mapping = {label: i for i, label in enumerate(le.classes_)}
    all_mappings[col] = mapping
    print(f"Column: {col}")
    for val, code in mapping.items():
        print(f"  {val} -> {code}")
    print()

# 2. YES_NO_COLS (manually encoded to 0/1)
# These were: ['site_head_neck', 'site_ul', 'site_ll', 'site_trunk', 'site_buttocks', 'symptomatic', 'macules', 'patch', 'papules', 'plaque', 'nodule', 'scales']
print("Column Type: Binary (Yes/No)")
binary_mapping = {"No/N": 0, "Yes/Y": 1}
for col in YES_NO_COLS:
    if col in df.columns:
        all_mappings[col] = binary_mapping
        print(f"  {col}: {binary_mapping}")

# 3. diagnosis_target (manually encoded)
print(f"\nColumn: diagnosis_target")
diagnosis_mapping = {"Non-MF": 0, "MF": 1}
all_mappings['diagnosis_target'] = diagnosis_mapping
print(f"  {diagnosis_mapping}")

print("\n--- TESTING MAPPINGS ---")

def verify_mapping(df_raw, df_encoded, col, mapping):
    print(f"Verifying {col}...")
    # Check a few non-null values
    sample_indices = df_encoded[col].dropna().head(5).index
    for idx in sample_indices:
        raw_val = str(df_raw.loc[idx, col]).strip().lower() if col in df_raw.columns else "N/A"
        encoded_val = df_encoded.loc[idx, col]
        
        # For LabelEncoded columns, the 'raw' value in encoders was normalized
        # For Binary columns, it was 'yes'/'no'
        
        # Simple check: does the encoded value match the mapping?
        match = False
        if col in encoders:
            # Re-normalize if needed (norm was used in earlier cells)
            # But here we can just check if le.inverse_transform works
            decoded = encoders[col].inverse_transform([int(encoded_val)])[0]
            print(f"  Row {idx}: Encoded={encoded_val} -> Decoded='{decoded}'")
        else:
            print(f"  Row {idx}: Encoded={encoded_val}")
            
# We'll use the 'raw' dataframe for comparison if needed, but since many values were normalized
# before encoding, we'll just show the sample transformations.
for col in list(encoders.keys())[:3]: # Test first 3 CAT_COLS
    verify_mapping(raw, df, col, all_mappings[col])

# Display value counts for all columns
print("\n--- VALUE COUNTS FOR ALL COLUMNS ---")
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False))
    print()

--- ENCODING MAPPINGS ---

Column: sex
  female -> 0
  male -> 1

Column: course
  intermittent -> 0
  progressive -> 1
  regressive -> 2
  remission and excerbation -> 3
  stationary -> 4
  unknown -> 5

Column: color
  erythematous -> 0
  hyperpigmented -> 1
  hypopigmented  -> 2
  poikilodermatous -> 3

Column: bx1_site
  buttocks -> 0
  face -> 1
  ll -> 2
  neck -> 3
  no_bx_site -> 4
  scalp -> 5
  trunk -> 6
  ul -> 7
  unknown -> 8

Column: bx2_site
  buttocks -> 0
  face -> 1
  ll -> 2
  neck -> 3
  no_bx_site -> 4
  trunk -> 5
  ul -> 6
  unknown -> 7

Column: bx1_morph
  macule -> 0
  no_bx_morph -> 1
  nodule -> 2
  papule   -> 3
  patch -> 4
  plaque -> 5

Column: bx2_morph
  macule -> 0
  no_bx_morph -> 1
  nodule -> 2
  papule  -> 3
  patch -> 4
  plaque -> 5

Column: current_ttt
  no -> 0
  yes -> 1

Column: visit_type
  FU -> 0
  New -> 1
  Rec -> 2

Column Type: Binary (Yes/No)
  site_head_neck: {'No/N': 0, 'Yes/Y': 1}
  site_ul: {'No/N': 0, 'Yes/Y': 1}
  site_ll: {'N